<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/05_applications/rag_with_reranking_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation with Re-Ranking

## Objective

Build a RAG pipeline that retrieves candidate documents
using embeddings and improves retrieval quality using
cross-encoder re-ranking before generating the final answer.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from transformers import pipeline

In [ ]:
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

In [ ]:
documents = [
    "Machine learning tutorials help beginners understand ML concepts.",
    "Deep learning uses neural networks to learn complex patterns.",
    "Natural language processing focuses on understanding text data.",
    "Python is widely used for machine learning and data science.",
    "Transformers are powerful models used in modern NLP systems."
]

In [ ]:
doc_embeddings = bi_encoder.encode(documents)

In [ ]:
query = "How can I start learning machine learning?"

query_embedding = bi_encoder.encode(query)

In [ ]:
scores = cosine_similarity([query_embedding], doc_embeddings)[0]

df = pd.DataFrame({
    "Document": documents,
    "Embedding Score": scores
}).sort_values(by="Embedding Score", ascending=False)

top_k = 3
candidates = df.head(top_k).copy()

In [ ]:
pairs = [[query, doc] for doc in candidates["Document"]]

rerank_scores = cross_encoder.predict(pairs)

candidates["Rerank Score"] = rerank_scores

candidates = candidates.sort_values(
    by="Rerank Score",
    ascending=False
)

In [ ]:
context = " ".join(candidates["Document"].tolist())

In [10]:
prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = generator(prompt, max_new_tokens=60)

print(response[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using the context below.

Context:
Machine learning tutorials help beginners understand ML concepts. Python is widely used for machine learning and data science. Deep learning uses neural networks to learn complex patterns.

Question:
How can I start learning machine learning?

Answer:
I started learning to learn Python in elementary school, and my friends were very interested in learning Python. I found it really helpful for them to learn Python and learn Python quickly. I also learned the basics of Python, the Python programming language, and the programming language.
This is just a few examples
